In [1]:
try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

In [2]:
if IN_COLAB:
  # Install dependencies
  ! pip install --upgrade pip
  ! pip install czitools
  ! pip install ipyfilechooser

In [3]:
# import the required libraries
from czitools.metadata_tools import czi_metadata as czimd
from czitools.utils import misc, planetable
from ipyfilechooser import FileChooser
from IPython.display import display, HTML
import os
import requests
import ipywidgets as widgets
import glob

### Define Parameters for Data Loading

In [ ]:
# try to find the folder with data and download otherwise from GitHub.

# Folder containing the input data
INPUT_FOLDER = './data/'

# Path to the data on GitHub
GITHUB_DATA_PATH = "https://media.githubusercontent.com/media/sebi06/ZEN_Python_Workshop/main/notebooks/data.zip"

# Download data
if not (os.path.isdir(INPUT_FOLDER)):
    import io
    import zipfile
    # Download training data
    compressed_data = './data.zip'
    if not os.path.isfile(compressed_data):
        print(f"Downloading data from: {GITHUB_DATA_PATH}")
        response = requests.get(GITHUB_DATA_PATH, allow_redirects=True)
        response.raise_for_status()

        # Diagnose if the response is not a real zip (e.g. LFS pointer or HTML error page)
        content = response.content
        print(f"Status: {response.status_code}, Content-Type: {response.headers.get('Content-Type')}, Size: {len(content)} bytes")
        if not content.startswith(b'PK'):
            raise ValueError(
                f"Downloaded content is not a zip file (missing PK header).\n"
                f"First 200 bytes: {content[:200]}"
            )

        compressed_data = io.BytesIO(content)

    with zipfile.ZipFile(compressed_data, 'r') as zip_accessor:
        zip_accessor.extractall('./')
        print(f"Extracted: {zip_accessor.namelist()}")


In [5]:
if not IN_COLAB:
    # choose local file
    fc = FileChooser()
    fc.default_path = INPUT_FOLDER
    fc.filter_pattern = '*.czi'
    display(fc)

elif IN_COLAB:
    # list files inside the folder on gdrive
    czifiles = glob.glob(os.path.join(INPUT_FOLDER, "*.czi"))
    wd = widgets.Select(
        options=czifiles,
        description='CZI Files:',
        layout={'width': 'max-content'}
    )
    display(wd)

FileChooser(path='F:\GitHub\ZEN_Python_Workshop\notebooks\data', filename='', title='', show_hidden=False, sel…

In [6]:
if not IN_COLAB:
    filepath = fc.selected
elif IN_COLAB:
    filepath = wd.value

print(f"Selected File: {filepath}")

Selected File: F:\GitHub\ZEN_Python_Workshop\notebooks\data\T=3_Z=5_CH=2_X=240_Y=170.czi


In [7]:
# get only specific metadata
czi_dimensions = czimd.CziDimensions(filepath)
print("SizeS: ", czi_dimensions.SizeS)
print("SizeT: ", czi_dimensions.SizeT)
print("SizeZ: ", czi_dimensions.SizeZ)
print("SizeC: ", czi_dimensions.SizeC)
print("SizeY: ", czi_dimensions.SizeY)
print("SizeX: ", czi_dimensions.SizeX)

SizeS:  None
SizeT:  3
SizeZ:  5
SizeC:  2
SizeY:  170
SizeX:  240


In [8]:
# and get more info
czi_scaling = czimd.CziScaling(filepath)
czi_channels = czimd.CziChannelInfo(filepath)
czi_bbox = czimd.CziBoundingBox(filepath)
czi_objectives = czimd.CziObjectives(filepath)
czi_detectors = czimd.CziDetector(filepath)
czi_microscope = czimd.CziMicroscope(filepath)
czi_sample = czimd.CziSampleInfo(filepath)

  0% |                                                  | ETA:  --:--:-- 0 of 30
100% |#################################################| Time:  0:00:00 30 of 30


In [9]:
# get the complete metadata at once as one big class
mdata = czimd.CziMetadata(filepath)

# convert metadata dictionary to a pandas dataframe
mdframe = misc.md2dataframe(mdata, reduced_params=True)

# create a ipywdiget to show the dataframe with the metadata
wd1 = widgets.Output(layout={"scrollY": "auto", "height": "300px"})

with wd1:
    display(HTML(mdframe.to_html()))
display(widgets.VBox(children=[wd1]))

  0% |                                                  | ETA:  --:--:-- 0 of 30
100% |#################################################| Time:  0:00:00 30 of 30


In [10]:
# write XML to disk
xmlfile = czimd.writexml(filepath)
print("XML File written to:", xmlfile)

XML File written to: F:\GitHub\ZEN_Python_Workshop\notebooks\data\T=3_Z=5_CH=2_X=240_Y=170_CZI_MetaData.xml


In [11]:
# get the planetable for the CZI file
pt, savepath = planetable.get_planetable(filepath,
                                         norm_time=True,
                                         save_table=True,
                                         planes={"time": 0, "channel": 0}
                                        )

# create a ipywdiget to show the dataframe with the metadata
wd2 = widgets.Output(layout={"scrollY": "auto", "height": "300px"})

with wd2:
    display(HTML(pt.to_html()))
display(widgets.VBox(children=[wd2]))

  0% |                                                  | ETA:  --:--:-- 0 of 30
100% |#################################################| Time:  0:00:00 30 of 30


2026-05-17 18:48:03,367 - czitools - INFO - Planetable saved successfully at: F:\GitHub\ZEN_Python_Workshop\notebooks\data\T=3_Z=5_CH=2_X=240_Y=170_planetable.csv
